In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Separate features (X) and target (y)
# Drop 'id' as it has no predictive power
X = train_df.drop(columns=['id', 'Irrigation_Need'])  # Drop target and id
y = train_df['Irrigation_Need']  # Target variable

# Preprocessing
# Encode the target variable (Low, Medium, High -> 0, 1, 2)
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

# Identify categorical columns and encode them for the models
categorical_cols = X.select_dtypes(include=['object']).columns

# Simple Ordinal Encoding for starter models
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set: {X_train.shape}, Validation set: {X_val.shape}")



Training set: (504000, 19), Validation set: (126000, 19)


In [2]:
# MODEL 2: BOOSTING (XGBoost)
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight

print("--- Starting Grid Search for XGBoost ---")

# 1. Calculate sample weights for the minority class (Critical for S6E4)
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# 2. Define the Grid
# Warning: Every item added here multiplies the total training time!
xgb_param_grid = {
    'n_estimators': [100, 200],           # 2 options
    'max_depth': [3, 6],                  # 2 options
    'learning_rate': [0.05, 0.1],         # 2 options
    'subsample': [0.8, 1.0]               # 2 options
}
# Total combinations: 2 * 2 * 2 * 2 = 16 combinations.
# With 3-fold CV, that is 48 total models to train.

# 3. Set up the base model
xgb_base = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    random_state=42,
    n_jobs=-1
)

# 4. Set up the Grid Search
xgb_grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=xgb_param_grid,
    scoring='balanced_accuracy',
    cv=3,
    verbose=1
)

# 5. Fit the grid search
# Note: sample_weight is passed to the fit method of the underlying estimator
xgb_grid_search.fit(X_train, y_train, sample_weight=sample_weights)

print(f"\nBest XGBoost Parameters: {xgb_grid_search.best_params_}")
print(f"Best CV Balanced Accuracy: {xgb_grid_search.best_score_:.4f}")

# Predict with the best model found
best_xgb_grid_model = xgb_grid_search.best_estimator_
xgb_grid_preds = best_xgb_grid_model.predict(X_val)

--- Starting Grid Search for XGBoost ---
Fitting 3 folds for each of 16 candidates, totalling 48 fits

Best XGBoost Parameters: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}
Best CV Balanced Accuracy: 0.9685


In [4]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

print("--- Preparing Test Data & Generating Submission ---")

# Load the Kaggle test data
test_df = pd.read_csv('test.csv')

# Define X_test by dropping the ID column
X_test = test_df.drop(columns=['id'])

# Preprocess X_test (Encode the categorical text columns into numbers)
categorical_cols = X_test.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    X_test[col] = le.fit_transform(X_test[col].astype(str))

# Grab the tuned model from your Grid Search
best_xgb_model = xgb_grid_search.best_estimator_

# Fit on the FULL training dataset
best_xgb_model.fit(X, y_encoded)

# Make predictions on our newly defined X_test
xgb_preds_encoded = best_xgb_model.predict(X_test) 

# Reverse the Label Encoding to get "Low", "Medium", "High"
xgb_preds_text = target_encoder.inverse_transform(xgb_preds_encoded)

# Build the Kaggle-formatted DataFrame
submission_xgb = pd.DataFrame({
    "id": test_df["id"],
    "Irrigation": xgb_preds_text
})

# Save to CSV
submission_xgb.to_csv("S6E4_xgb_submission.csv", index=False)
print("Success! XGBoost submission saved!")

--- Preparing Test Data & Generating Submission ---
Success! XGBoost submission saved!
